In [77]:
import itertools
import sympy as sp
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from finite_groups import FiniteGroup
from induced_rep_solver import InducedRepSolver
from symchar.symchar import character_table
from symchar.symchar.partition import generate_partitions


def RELU(x): return sp.Max(0, x)
def TANH(x): return sp.tanh(x)
def SIGMOID(x): return 1 / (1 + sp.exp(-x))
def LINEAR(x): return x

# Utils

In [78]:
def build_Cn(n):

    elements = list(range(n))

    def mult(a,b):
        return (a+b) % n

    return FiniteGroup(elements, mult)


def build_Sk(k):

    elements = list(itertools.permutations(range(k)))

    def mult(a,b):
        return tuple(a[i] for i in b)

    return FiniteGroup(elements, mult)


def build_product(G1,G2):

    elements = [(g,h) for g in G1.elements for h in G2.elements]

    def mult(x,y):

        g1,h1 = x
        g2,h2 = y

        return (
            G1.mult_func(g1,g2),
            G2.mult_func(h1,h2)
        )

    return FiniteGroup(elements, mult)


def build_H(k):

    perms = list(itertools.permutations(range(k-1)))

    H = []

    for p in perms:

        perm = list(range(k))

        for i in range(k-1):
            perm[i] = p[i]

        H.append((0,tuple(perm)))

    return H




## Characters

In [110]:
import numpy as np
import sympy as sp


def Sk_character_table(k):
    """
    Returns:
        char_table : (#classes × #irreps)
        labels     : partition labels
    """

    char_table = np.array(character_table(k), dtype=object)

    labels = [''.join(map(str, p)) for p in generate_partitions(k)]

    return char_table, labels


def Cn_character_table(n):
    table = []
    for g in range(n):
        row = []
        for j in range(n):
            row.append(sp.simplify(sp.exp(2 * sp.pi * sp.I * j * g / n)))
        table.append(row)
    labels = [f"chi{j}" for j in range(n)]
    return np.array(table, dtype=object), labels


def product_class_character_map_from_tables(G, Cn_table, Cn_labels, Sk_table, Sk_labels):
    """
    Build the class_character_map expected by solver.load_character_table(...)

    Output:
        class_character_map[cls.representative] = list of values
        labels = list of irrep labels for C_n x S_k
    """

    # --- identify the S_k conjugacy classes inside G.classes ---
    # Each cls.representative is of the form (c_elem, sigma)
    # Since C_n is abelian, the class of (c,sigma) is determined by c and the S_k class of sigma.

    # Build a lookup from an S_k representative to its row index in the supplied Sk_table.
    # We assume the supplied Sk_table rows are in the same order as generate_partitions(k),
    # and that the user can also supply a function/class map if needed.
    #
    # More robustly, map by cycle type:
    def cycle_type(perm):
        k = len(perm)
        seen = [False] * k
        parts = []
        for i in range(k):
            if not seen[i]:
                length = 0
                j = i
                while not seen[j]:
                    seen[j] = True
                    j = perm[j]
                    length += 1
                parts.append(length)
        return tuple(sorted(parts, reverse=True))

    # infer row lookup for S_k by partition label
    # Example label '311' -> partition (3,1,1)
    sk_partition_to_row = {}
    for row_idx, lbl in enumerate(Sk_labels):
        part = tuple(int(ch) for ch in lbl)
        sk_partition_to_row[part] = row_idx

    class_character_map = {}

    for cls in G.classes:
        c_elem, sigma = cls.representative

        sigma_part = cycle_type(sigma)
        if sigma_part not in sk_partition_to_row:
            raise ValueError(
                f"No row in Sk_table matches cycle type {sigma_part} of class representative {sigma}"
            )
        sk_row = sk_partition_to_row[sigma_part]

        values = []
        for i in range(len(Cn_labels)):
            for j in range(len(Sk_labels)):
                values.append(sp.simplify(Cn_table[c_elem, i] * Sk_table[sk_row, j]))

        class_character_map[cls.representative] = values

    labels = [f"{c}⊗{s}" for c in Cn_labels for s in Sk_labels]
    return class_character_map, labels

In [117]:
n = 3
k = 4

In [116]:
char_table

{(0, (0, 1, 2, 3)): [-1, 1, 1, -1, 1, -1, 1, 1, -1, 1],
 (0, (0, 1, 3, 2)): [1, 0, -1, -1, 3, 1, 0, -1, -1, 3],
 (0, (0, 2, 3, 1)): [-1, 0, -1, 1, 3, -1, 0, -1, 1, 3],
 (0, (1, 0, 3, 2)): [0, -1, 2, 0, 2, 0, -1, 2, 0, 2],
 (0, (1, 2, 3, 0)): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 (1, (0, 1, 2, 3)): [-1, 1, 1, -1, 1, 1, -1, -1, 1, -1],
 (1, (0, 1, 3, 2)): [1, 0, -1, -1, 3, -1, 0, 1, 1, -3],
 (1, (0, 2, 3, 1)): [-1, 0, -1, 1, 3, 1, 0, 1, -1, -3],
 (1, (1, 0, 3, 2)): [0, -1, 2, 0, 2, 0, 1, -2, 0, -2],
 (1, (1, 2, 3, 0)): [1, 1, 1, 1, 1, -1, -1, -1, -1, -1]}

In [ ]:
Cn = build_Cn(n)
Sk = build_Sk(k)

G = build_product(Cn,Sk)
H = build_H(k)

solver = InducedRepSolver(G)
solver.set_subgroup(H)

Sk_char_table, Sk_labels = Sk_character_table(k)
Cn_char_table, Cn_labels = Cn_character_table(n)
char_table, labels = product_class_character_map_from_tables(G, Cn_char_table, Cn_labels, Sk_char_table, Sk_labels)
solver.load_character_table(char_table,labels)

solver.compute_projectors(refine=False)

graph = solver.build_interaction_graph(RELU)

solver.visualise_interaction_graph(graph, group_name=f"C{n} x S{k}")

Building multiplication table for G (order 3)...
Discovering conjugacy classes...
Building multiplication table for G (order 24)...
Discovering conjugacy classes...
Building multiplication table for G (order 72)...
Discovering conjugacy classes...


RuntimeError: Irrep matrices required for refinement.

In [107]:
labels

['chi0^3', 'chi0^21', 'chi0^111', 'chi1^3', 'chi1^21', 'chi1^111']